# Stage A — supervised pretraining of the PlayerConsolidator

Branches off L2 (`CrossEntityAttention.ctx_now`) and trains a new `PlayerConsolidator` block that emits one strategic embedding per player slot (`player_state (B, 4, d)`). The Consolidator is the only trainable module here:

- **Frozen**: L0 (planet + fleet + comet), L1 (`PlanetEntityEncoder`), L2 (`CrossEntityAttention`), L3 (`DualRoleAttention`), L4 (`JointRoleAttention`), `PairHead`.
- **Trainable**: `entity_model.consolidator` + `CurrentStateHead` (per-task MLPs ≥ 2 layers).

## v0 supervision (this notebook): CurrentStateHead only

Per-player labels are derived from the current snapshot — no cache rebuild needed. The head emits 5 continuous channels (`total_ships_log`, `planet_count`, `production_sum`, `fleet_ships_in_air_log`, `comet_count`) plus `alive` (BCE) and `rank` (4-way CE).

v1+ heads (`FutureStateHead`, `OutcomeHead`, `MatchupHead`) need future-snapshot or per-episode terminal labels and are deferred.

## What this notebook does **not** do

- No PPO critic wiring (the PPO `value_head` still reads `glob` from L2; switch will happen after this stage validates).
- No JEPA, no action-conditioned transition, no actor-side FiLM-on-`player_state`.

## Inputs from GCS

```
gs://orbit-wars-shipping/entity/
  code.tgz                                                    # agents/ + scripts/build_pair_dataset_orbital_occle.py
  weights.tgz                                                 # frozen L0 (d=256)
  pair_cache.pt                                               # snapshot source (pair labels ignored)
  runs/top4_pair2head_film_d256_h8_lr5e-05_b256_30ep_.../     # warm-start L1-L4 + PairHead
```


## 1. Authenticate + pull bundle from GCS

In [ ]:
from google.colab import auth
auth.authenticate_user()
BUCKET = 'gs://orbit-wars-shipping/entity'
print(f'pulling from {BUCKET}')

In [ ]:
import os, subprocess, time, hashlib, json, concurrent.futures
from pathlib import Path

WORK = Path('/content/orbit-wars')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

# Stage A doesn't need acted/pair labels — every snapshot has valid
# per-player current-state labels. The 13 GB single-file Ebi cache is
# fine; switch to the chunked top4 cache if you want more diversity.
PAIR_CACHE_PREFIX = 'pair_cache'

def _gcs_size(url: str):
    try:
        out = subprocess.run(
            ['gcloud', 'storage', 'objects', 'describe', url,
             '--format=value(size)'],
            check=True, capture_output=True, text=True,
        )
    except subprocess.CalledProcessError:
        return None
    try:
        return int(out.stdout.strip())
    except (TypeError, ValueError):
        return None

def _gcloud_cp(src, dst, *, force=False, quiet=True):
    if dst.exists():
        if not force:
            return dst.name, 0.0, dst.stat().st_size
        dst.unlink()
    cmd = ['gcloud', 'storage', 'cp', src, str(dst)]
    if quiet:
        cmd.append('--quiet')
    else:
        print(f'  pulling {src}  →  {dst.name} ...', flush=True)
    t0 = time.time()
    subprocess.run(cmd, check=True)
    return dst.name, time.time() - t0, dst.stat().st_size

SMALL_TASKS = [
    (f'{BUCKET}/code.tgz',    WORK / 'code.tgz',    True),
    (f'{BUCKET}/weights.tgz', WORK / 'weights.tgz', True),
]

PAIR_CACHE = WORK / 'pair_cache.pt'
MANIFEST_LOCAL = WORK / f'{PAIR_CACHE_PREFIX}.manifest.json'

def _fetch_manifest():
    if _gcs_size(f'{BUCKET}/{PAIR_CACHE_PREFIX}.manifest.json') is None:
        return None
    subprocess.run(
        ['gcloud', 'storage', 'cp', '--quiet',
         f'{BUCKET}/{PAIR_CACHE_PREFIX}.manifest.json',
         str(MANIFEST_LOCAL)],
        check=True,
    )
    return json.loads(MANIFEST_LOCAL.read_text())

def _sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for blk in iter(lambda: fh.read(1 << 20), b''):
            h.update(blk)
    return h.hexdigest()

def _pull_chunk(spec, chunks_dir):
    name = spec['name']
    dst = chunks_dir / name
    expected_size = int(spec.get('size_bytes', 0))
    expected_sha = spec.get('sha256')
    if (
        dst.exists()
        and (expected_size == 0 or dst.stat().st_size == expected_size)
        and (expected_sha is None or _sha256_file(dst) == expected_sha)
    ):
        return name, 0.0, dst.stat().st_size
    if dst.exists():
        dst.unlink()
    t0 = time.time()
    print(f'    pulling chunk {name} ...', flush=True)
    subprocess.run(
        ['gcloud', 'storage', 'cp', f'{BUCKET}/{name}', str(dst)],
        check=True,
    )
    if expected_sha and _sha256_file(dst) != expected_sha:
        raise RuntimeError(f'sha256 mismatch on chunk {name}; refusing.')
    return name, time.time() - t0, dst.stat().st_size

def _assemble_chunks(manifest, chunks_dir):
    chunks = [c['name'] for c in manifest['chunks']]
    total_bytes = int(manifest.get('total_bytes', 0))
    print(f'  assembling {len(chunks)} chunks → pair_cache.pt ...')
    t0 = time.time()
    if PAIR_CACHE.exists():
        PAIR_CACHE.unlink()
    with open(PAIR_CACHE, 'wb') as out_fh:
        for name in chunks:
            with open(chunks_dir / name, 'rb') as in_fh:
                while True:
                    blk = in_fh.read(1 << 22)
                    if not blk:
                        break
                    out_fh.write(blk)
    actual = PAIR_CACHE.stat().st_size
    print(f'  assembled in {time.time()-t0:.1f}s  ({actual/1024**3:.2f} GB)')
    if total_bytes and actual != total_bytes:
        raise RuntimeError(
            f'assembly size mismatch: got {actual} bytes, '
            f'manifest declares {total_bytes}.'
        )
    return PAIR_CACHE

def _pull_pair_cache():
    t0 = time.time()
    manifest = _fetch_manifest()
    if manifest is None:
        for cand in (f'{PAIR_CACHE_PREFIX}.pt', 'pair_cache.pt'):
            remote_size = _gcs_size(f'{BUCKET}/{cand}')
            if remote_size is None:
                continue
            if PAIR_CACHE.exists() and PAIR_CACHE.stat().st_size == remote_size:
                return f'{cand} (single, cached)', 0.0, remote_size
            if PAIR_CACHE.exists():
                PAIR_CACHE.unlink()
            name, dt, size = _gcloud_cp(
                f'{BUCKET}/{cand}', PAIR_CACHE, force=False, quiet=False,
            )
            return f'{cand} (single)', dt, size
        raise RuntimeError(
            f'no pair cache found on {BUCKET}: tried '
            f'{PAIR_CACHE_PREFIX}.manifest.json, {PAIR_CACHE_PREFIX}.pt, '
            f'pair_cache.pt'
        )
    total = int(manifest.get('total_bytes', 0))
    if PAIR_CACHE.exists() and total and PAIR_CACHE.stat().st_size == total:
        return 'pair_cache.pt (chunked, cached)', 0.0, PAIR_CACHE.stat().st_size
    chunks_dir = WORK / f'{PAIR_CACHE_PREFIX}_chunks'
    chunks_dir.mkdir(parents=True, exist_ok=True)
    chunk_specs = manifest['chunks']
    print(f'  chunked pair cache: {len(chunk_specs)} chunks, {total/1024**3:.2f} GB total')
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(chunk_specs)) as pool:
        futures = {pool.submit(_pull_chunk, spec, chunks_dir): spec['name'] for spec in chunk_specs}
        for fut in concurrent.futures.as_completed(futures):
            cname, cdt, csize = fut.result()
            mb = csize / 1024 / 1024
            if cdt == 0.0:
                print(f'    {cname:<32s} {mb:>7.1f} MB  (cached)')
            else:
                print(f'    {cname:<32s} {mb:>7.1f} MB  in {cdt:5.1f}s  ({mb/max(cdt, 1e-3):6.1f} MB/s)')
    _assemble_chunks(manifest, chunks_dir)
    return f'pair_cache.pt (chunked, {len(chunk_specs)})', time.time() - t0, PAIR_CACHE.stat().st_size

T_START = time.time()
all_tasks = []
with concurrent.futures.ThreadPoolExecutor(max_workers=len(SMALL_TASKS) + 1) as pool:
    futures = []
    for src, dst, force in SMALL_TASKS:
        futures.append(pool.submit(_gcloud_cp, src, dst, force=force, quiet=True))
    futures.append(pool.submit(_pull_pair_cache))
    for fut in concurrent.futures.as_completed(futures):
        all_tasks.append(fut.result())

for label, dt, size in sorted(all_tasks, key=lambda t: -t[2]):
    mb = size / 1024 / 1024
    if dt == 0.0:
        print(f'  {label:<32s} {mb:>9.1f} MB  (cached, skipped)')
    else:
        print(f'  {label:<32s} {mb:>9.1f} MB  in {dt:5.1f}s  ({mb/max(dt, 1e-3):6.1f} MB/s)')

print(f'\npair_cache.pt: {PAIR_CACHE.stat().st_size/1024**3:.2f} GB  (total wall: {time.time()-T_START:.1f}s)')

In [ ]:
# Wipe stale extracted code so a previous bundle can't silently shadow
# the new one. The pair_cache.pt is left alone (huge file).
!rm -rf agents scripts ckpts data
!find . -maxdepth 1 -name '*.pt' ! -name 'pair_cache.pt' -delete

!tar xzf code.tgz
!tar xzf weights.tgz

import sys
for m in list(sys.modules):
    if m.startswith('agents') or m.startswith('scripts'):
        del sys.modules[m]
import importlib, gc
importlib.invalidate_caches()
gc.collect()
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null || true
!ls -la

## 1b. Verify the bundle is the consolidator-aware version

Fails fast if `code.tgz` is from before the Consolidator landed.

In [ ]:
import torch, agents
from agents.transformer_v2.aggregator import PlayerConsolidator
from agents.transformer_v2.pretrain.consolidator_heads import (
    CurrentStateHead, compute_current_state_labels, current_state_loss,
)
from agents.transformer_v2.pretrain.entity_encoder import EntityPretrainModel

print(f'agents module file: {agents.__file__}')
assert 'Minimal' in (agents.__doc__ or ''), 'agents/__init__.py is NOT the bundle shim — stale extract.'

m = EntityPretrainModel(d_model=256, n_steps=6, conditioner_n_layers=3, head_n_layers=3)
assert hasattr(m, 'consolidator'), 'EntityPretrainModel missing .consolidator — stale code.tgz (pre-consolidator).'
assert isinstance(m.consolidator, PlayerConsolidator)

# forward_with_context returns player_state
B, P, F = 2, 8, 12
planet_tokens = torch.randn(B, P, 256)
fleet_tokens = torch.randn(B, F, 256)
planet_mask = torch.ones(B, P, dtype=torch.bool)
owner_oh = torch.zeros(B, P, 5)
owner_oh[:, :4, :4] = torch.eye(4)
owner_oh[:, 4:, 4] = 1.0
routing = {
    'fleet_target_idx': torch.zeros(B, F, dtype=torch.long),
    'fleet_source_idx': torch.zeros(B, F, dtype=torch.long),
    'fleet_owner_slot': torch.zeros(B, F, dtype=torch.long),
    'fleet_ships_log': torch.zeros(B, F),
    'fleet_eta_norm': torch.zeros(B, F),
    'fleet_mask': torch.zeros(B, F, dtype=torch.bool),
}
out = m.forward_with_context(planet_tokens, fleet_tokens, routing, planet_mask, planet_owner_oh=owner_oh)
assert 'player_state' in out
assert out['player_state'].shape == (B, 4, 256), out['player_state'].shape

# CurrentStateHead defaults: trunk=2 layers, per-task head=2 layers
head = CurrentStateHead(d_model=256)
trunk_lin = sum(1 for x in head.trunk if isinstance(x, torch.nn.Linear))
cont_lin = sum(1 for x in head.continuous_head if isinstance(x, torch.nn.Linear))
assert trunk_lin >= 2, trunk_lin
assert cont_lin >= 2, cont_lin
preds = head(out['player_state'])
assert preds['continuous'].shape == (B, 4, 5)
assert preds['alive_logits'].shape == (B, 4)
assert preds['rank_logits'].shape == (B, 4, 4)

n_cons = sum(p.numel() for p in m.consolidator.parameters())
n_head = sum(p.numel() for p in head.parameters())
print(f'PlayerConsolidator params:  {n_cons:>9,}')
print(f'CurrentStateHead params:    {n_head:>9,}')
print('verify OK')

## 2. Stage the pair cache into the layout the CLI expects

Stage A's CLI takes `--pair-cache-path` directly, but we keep the directory-and-filename convention so the cached features are findable by other scripts.

In [ ]:
from pathlib import Path
import os

PAIR_CACHE_LAYOUT = {
    'pair_cache_top4': (
        'bowwowforeach_Ebi_ErfanEshratifar_Shun_PI_T6',
        'bowwowforeach_Ebi_ErfanEshratifar_Shun_PI_T6_p64_f1024_all.pt',
    ),
    'pair_cache': (
        'bowwowforeach_Ebi_T6',
        'bowwowforeach_Ebi_T6_p64_f1024_all.pt',
    ),
}
CACHE_SUBDIR, CACHE_FILENAME = PAIR_CACHE_LAYOUT.get(
    PAIR_CACHE_PREFIX, PAIR_CACHE_LAYOUT['pair_cache'],
)
CACHE_DIR = Path(f'data/datasets/_pair_cache/{CACHE_SUBDIR}')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TARGET = CACHE_DIR / CACHE_FILENAME
SOURCE = Path('/content/orbit-wars/pair_cache.pt')
if CACHE_TARGET.exists():
    CACHE_TARGET.unlink()
os.link(SOURCE, CACHE_TARGET)
size_gb = CACHE_TARGET.stat().st_size / 1024**3
print(f'staged: {CACHE_TARGET}')
print(f'        {size_gb:.2f} GB')
PAIR_CACHE_PATH = str(CACHE_TARGET)

## 3. Verify torch + GPU

In [ ]:
import torch
print(f'torch: {torch.__version__}, cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  device: {torch.cuda.get_device_name(0)}')
    print(f'  mem total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 4. Stage the frozen L0 ckpts into the CLI's `--*-run-dir` layout

In [ ]:
import shutil
from pathlib import Path
PLANET_RUN_DIR = Path('/content/orbit-wars/ckpts/planet')
FLEET_RUN_DIR  = Path('/content/orbit-wars/ckpts/fleet')
COMET_RUN_DIR  = Path('/content/orbit-wars/ckpts/comet')
for d in (PLANET_RUN_DIR, FLEET_RUN_DIR, COMET_RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/orbit-wars/planet_encoder_best.pt', PLANET_RUN_DIR / 'planet_encoder_best.pt')
shutil.copy('/content/orbit-wars/fleet_encoder_best.pt',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt')
shutil.copy('/content/orbit-wars/comet_past_best.pt',     COMET_RUN_DIR  / 'comet_past_best.pt')
import torch
for tag, p in (('planet', PLANET_RUN_DIR / 'planet_encoder_best.pt'),
                ('fleet',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt'),
                ('comet',  COMET_RUN_DIR  / 'comet_past_best.pt')):
    c = torch.load(p, map_location='cpu', weights_only=False)
    print(f'{tag:6s} ckpt: d_model={c["config"]["d_model"]}, epoch={c["epoch"]}')
    assert c['config']['d_model'] == 256

## 4b. Pull the baseline entity ckpt (warm-start for frozen L1–L4 + PairHead)

The Consolidator is **not** in this ckpt (it's a new module). When loading with `strict=False`, the Consolidator's keys are reported as missing and stay at fresh init — exactly what we want.

In [ ]:
import subprocess
from pathlib import Path
BASELINE_RUN = 'top4_pair2head_film_d256_h8_lr5e-05_b256_30ep_20260521-003000'
BASELINE_DIR = Path(f'/content/orbit-wars/ckpts/baseline/{BASELINE_RUN}')
BASELINE_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_CKPT = BASELINE_DIR / 'entity_encoder_best.pt'
if not BASELINE_CKPT.exists():
    subprocess.run(
        ['gcloud', 'storage', 'cp',
         f'{BUCKET}/runs/{BASELINE_RUN}/entity_encoder_best.pt',
         str(BASELINE_CKPT)],
        check=True,
    )
print(f'baseline ckpt: {BASELINE_CKPT}  ({BASELINE_CKPT.stat().st_size/1024**2:.1f} MB)')
import torch
saved = torch.load(BASELINE_CKPT, map_location='cpu', weights_only=False)
cfg = saved.get('config', {})
print(f'  saved epoch:        {saved.get("epoch")}')
print(f'  saved d_model:      {cfg.get("d_model")}')
print(f'  saved n_steps:      {cfg.get("n_steps")}')
print(f'  saved conditioner:  {cfg.get("conditioner_n_layers", 1)} layers')
print(f'  saved head_n_layers:{cfg.get("head_n_layers", 1)}')

## 5. Train Stage A

The CLI freezes everything except `entity_model.consolidator` + `CurrentStateHead`. Per-batch flow:

1. Frozen L0 forward (planet / fleet / comet specialists).
2. Frozen L1 + L2 → `ctx_now (B, P, 256)`.
3. `PlayerConsolidator(ctx_now, planet_mask, owner_oh)` → `player_state (B, 4, 256)`.
4. `CurrentStateHead(player_state)` → continuous (5) + alive logit + rank logit.
5. Labels from raw `planet_features` / `fleet_features` → Huber + BCE + CE loss.

In [ ]:
D_MODEL                       = 256
CROSS_N_LAYERS                = 2
CONDITIONER_N_LAYERS          = 3      # frozen, kept consistent with baseline ckpt
HEAD_N_LAYERS                 = 3      # frozen, kept consistent with baseline ckpt
STATE_HEAD_TRUNK_N_LAYERS     = 2
STATE_HEAD_N_LAYERS           = 2      # per-task MLP depth (floor 2)
BATCH_SIZE                    = 64     # frozen perception + small head — lots of headroom on T4
EPOCHS                        = 10
LR                            = 1e-4
WEIGHT_DECAY                  = 1e-4
VAL_FRAC                      = 0.10
TEST_FRAC                     = 0.10
SEED                          = 1729
DEVICE                        = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
INIT_FROM_ENTITY              = str(BASELINE_CKPT)

import time
TS = time.strftime('%Y%m%d-%H%M%S')
RUN_TAG = f'consolidator_v0_current_state_d{D_MODEL}_th{STATE_HEAD_TRUNK_N_LAYERS}_hh{STATE_HEAD_N_LAYERS}_lr{LR:g}_b{BATCH_SIZE}_{EPOCHS}ep_{TS}'
OUT_DIR = f'data/runs/consolidator/{RUN_TAG}'
print('out dir:', OUT_DIR)
print(f'warm-start: {INIT_FROM_ENTITY}')

In [ ]:
!python -u -m agents.transformer_v2.pretrain.train_consolidator \
  --planet-run-dir $PLANET_RUN_DIR \
  --fleet-run-dir  $FLEET_RUN_DIR \
  --comet-run-dir  $COMET_RUN_DIR \
  --pair-cache-path $PAIR_CACHE_PATH \
  --init-from-entity-ckpt $INIT_FROM_ENTITY \
  --out-dir $OUT_DIR \
  --d-model $D_MODEL \
  --cross-n-layers $CROSS_N_LAYERS \
  --conditioner-n-layers $CONDITIONER_N_LAYERS \
  --head-n-layers $HEAD_N_LAYERS \
  --state-head-trunk-n-layers $STATE_HEAD_TRUNK_N_LAYERS \
  --state-head-n-layers $STATE_HEAD_N_LAYERS \
  --batch-size $BATCH_SIZE \
  --epochs $EPOCHS \
  --lr $LR \
  --weight-decay $WEIGHT_DECAY \
  --val-frac $VAL_FRAC \
  --test-frac $TEST_FRAC \
  --seed $SEED \
  --device $DEVICE

## 6. Push the trained Stage A run back to GCS

In [ ]:
import subprocess
from pathlib import Path
src = Path(OUT_DIR)
assert src.is_dir(), src
# Land Stage A runs under their own subdir so they don't collide with
# the entity_encoder runs above.
dst_parent = f'{BUCKET}/runs/consolidator/'
subprocess.run(
    ['gcloud', 'storage', 'cp', '--recursive', str(src), dst_parent],
    check=True,
)
print(f'uploaded to: {dst_parent}{src.name}/')
subprocess.run(
    ['gcloud', 'storage', 'ls', '--long', '--readable-sizes',
     f'{dst_parent}{src.name}/'],
    check=False,
)